# Embeddings and attention

**Track:** Training LLMs. **Level:** Intermediate. **CPU:** yes.

## Goals

- Build word vectors from a co-occurrence matrix and check a nearest-neighbor relation you can see in the corpus.
- Implement scaled dot-product attention, a causal mask, and a padding mask.
- See what a temperature does to the attention weights.

## Sources and license

Original notebook. The embedding section follows the co-occurrence / PPMI / SVD line of Levy and Goldberg and of the word2vec papers, which is also how Stanford CS224N introduces distributional vectors. CS224N assignment code is not copied. Attention is the scaled dot-product from Vaswani et al., *Attention Is All You Need* (2017). The formulas are implemented from the paper, not from a course repository.

## Concepts

A word vector here is a low-rank view of "which words appear nearby." PPMI up-weights pairs that occur more than independence would predict and clips negative values at zero. SVD keeps the directions with the largest singular values. Attention is a weighted average of value vectors. The weights are a softmax of query-key dot products, divided by `sqrt(d)` so the variance of the scores does not grow with the key dimension. A causal mask sets future scores to `-inf` before the softmax. A padding mask does the same for positions that are not real tokens.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'torch'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import math
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from labs._shared.seed import seed_everything

seed_everything(0)


## Guided exercise: co-occurrence embeddings

The corpus repeats two topics. `river` should land nearer `water` than `money`. `loan` should land nearer `interest` than `fish`. If either comparison fails, the window or the corpus is wrong, not the assertion.


In [ ]:
sentences = [
    "river water bank fish",
    "river water flow fish",
    "bank money account loan",
    "bank money interest account",
    "fish swim river water",
    "loan interest money account",
] * 8
vocab = sorted({word for sentence in sentences for word in sentence.split()})
index = {word: i for i, word in enumerate(vocab)}
counts = np.zeros((len(vocab), len(vocab)))
for sentence in sentences:
    tokens = sentence.split()
    for i, word in enumerate(tokens):
        for j in range(max(0, i - 2), min(len(tokens), i + 3)):
            if i != j:
                counts[index[word], index[tokens[j]]] += 1
row = counts.sum(axis=1, keepdims=True)
col = counts.sum(axis=0, keepdims=True)
total = counts.sum()
expected = row @ col / total
ppmi = np.zeros_like(counts)
mask = counts > 0
ppmi[mask] = np.maximum(np.log(counts[mask] / expected[mask]), 0.0)
u, s, _vt = np.linalg.svd(ppmi, full_matrices=False)
vectors = u[:, :4] * s[:4]

def cosine(a, b):
    va, vb = vectors[index[a]], vectors[index[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-12))

print("river~water", round(cosine("river", "water"), 3), "river~money", round(cosine("river", "money"), 3))
print("loan~interest", round(cosine("loan", "interest"), 3), "loan~fish", round(cosine("loan", "fish"), 3))
assert cosine("river", "water") > cosine("river", "money")
assert cosine("loan", "interest") > cosine("loan", "fish")


## Guided exercise: scaled dot-product attention

`Q`, `K`, and `V` are shaped `(batch, time, dim)`. The causal mask is a lower triangle. Each row of the weight matrix sums to 1, and position 0 puts no mass on position 2.


In [ ]:
def attention(query, key, value, causal=False, key_padding=None):
    scores = query @ key.transpose(-2, -1) / math.sqrt(query.shape[-1])
    if causal:
        t = query.shape[-2]
        causal_mask = torch.tril(torch.ones(t, t, dtype=torch.bool))
        scores = scores.masked_fill(~causal_mask, float("-inf"))
    if key_padding is not None:
        # key_padding True means "this key position is padding".
        scores = scores.masked_fill(key_padding[:, None, :], float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ value, weights

torch.manual_seed(0)
q = torch.randn(1, 3, 4)
k = torch.randn(1, 3, 4)
v = torch.randn(1, 3, 4)
out, weights = attention(q, k, v, causal=True)
print(weights[0].round(decimals=3))
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, 3), atol=1e-5)
assert float(weights[0, 0, 2]) == 0.0


## Your turn

Pass a padding mask that marks the last key as padding. Its column of attention weights must be zero, and the remaining weights in each row must still sum to 1.

## Solution


In [ ]:
padding = torch.tensor([[False, False, True]])
_out, padded = attention(q, k, v, causal=False, key_padding=padding)
print(padded[0].round(decimals=3))
assert torch.allclose(padded[0, :, 2], torch.zeros(3), atol=1e-6)
assert torch.allclose(padded.sum(dim=-1), torch.ones(1, 3), atol=1e-5)


## Beyond the course

Divide the scores by a temperature before the softmax. A larger temperature flattens the distribution, so the maximum weight falls. This is the same knob as the temperature in a language-model sampler. It is not a new parameter of the architecture.


In [ ]:
def max_weight(temperature):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    return float(torch.softmax(scores / temperature, dim=-1).max())

cool, hot = max_weight(1.0), max_weight(4.0)
print("max weight T=1", round(cool, 3), "T=4", round(hot, 3))
assert cool > hot
print("LAB OK")


## Research questions

1. `bank` occurred in both topics. Which neighbor did it actually get, and what would a sense-specific vector have to change in the count matrix?
2. The causal mask and the padding mask both write `-inf` into `scores`. What happens to a row whose keys are all masked?
3. Vaswani et al. scale by `sqrt(d)`. If you remove the scale and increase `d` from 4 to 64, what happens to the softmax before any training?

## Further reading

- Vaswani et al., *Attention Is All You Need* (2017).
- Levy and Goldberg, *Neural Word Embedding as Implicit Matrix Factorization* (2014).
- Mikolov et al., *Distributed Representations of Words and Phrases and their Compositionality* (2013).
- Stanford CS224N lecture notes on distributional semantics and on Transformers. The assignment code is not copied. https://web.stanford.edu/class/cs224n/
